# import 

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 데이터 로드

In [2]:
# 0) 데이터 로드
df = pd.read_csv("diabetes_prediction_dataset.csv", encoding="cp949")
df.info() # 데이터 타입 확인 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   성별        100000 non-null  object 
 1   나이        100000 non-null  float64
 2   고혈압 여부    100000 non-null  int64  
 3   심장질환 여부   100000 non-null  int64  
 4   흡연 경험     100000 non-null  object 
 5   BMI 지수    100000 non-null  float64
 6   당화혈색소 수치  100000 non-null  float64
 7   혈당 수치     100000 non-null  int64  
 8   당뇨병 여부    100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


# 데이터 분할 

Train:Valid:Test=7:2:1

In [3]:
print("\n" + "="*60)
print("데이터 분할: Train(70%) : Valid(20%) : Test(10%)")
print("="*60)

target_col = "당뇨병 여부"

# 1단계: Train+Valid(90%) vs Test(10%)
train_val_df, test_df = train_test_split(
    df, 
    test_size=0.1,  
    random_state=42, 
    stratify=df[target_col]
)

# 2단계: Train(70%) vs Valid(20%)
train_df, valid_df = train_test_split(
    train_val_df,
    test_size=0.2222222222,  
    random_state=42,
    stratify=train_val_df[target_col]
)

print(f"\nTrain: {len(train_df):,}개 ({len(train_df)/len(df)*100:.1f}%)")
print(f"Valid: {len(valid_df):,}개 ({len(valid_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df):,}개 ({len(test_df)/len(df)*100:.1f}%)")

# 타겟 분포 확인
print(f"\n타겟 변수 분포:")
print(f"Train - 0: {(train_df[target_col]==0).sum()/len(train_df)*100:.2f}%, 1: {(train_df[target_col]==1).sum()/len(train_df)*100:.2f}%")
print(f"Valid - 0: {(valid_df[target_col]==0).sum()/len(valid_df)*100:.2f}%, 1: {(valid_df[target_col]==1).sum()/len(valid_df)*100:.2f}%")
print(f"Test  - 0: {(test_df[target_col]==0).sum()/len(test_df)*100:.2f}%, 1: {(test_df[target_col]==1).sum()/len(test_df)*100:.2f}%")

# X, y 분리
X_train_raw = train_df.drop(target_col, axis=1).copy()
y_train = train_df[target_col].copy()

X_valid_raw = valid_df.drop(target_col, axis=1).copy()
y_valid = valid_df[target_col].copy()

X_test_raw = test_df.drop(target_col, axis=1).copy()
y_test = test_df[target_col].copy()

# 1. 데이터 크기(행, 열) 확인: 데이터가 의도대로 잘 쪼개졌는지 체크
print(f"\n학습용 데이터 크기: {X_train_raw.shape}, 정답: {y_train.shape}")
print(f"검증용 데이터 크기: {X_valid_raw.shape}, 정답: {y_valid.shape}")
print(f"테스트용 데이터 크기: {X_test_raw.shape}, 정답: {y_test.shape}")

# 2. 타깃(종속변수) 비율 확인: stratify가 잘 작동해서 비율이 깨지지 않았는지 체크
print(f"\n학습 데이터 타깃 비율:\n{y_train.value_counts(normalize=True)}")


데이터 분할: Train(70%) : Valid(20%) : Test(10%)

Train: 70,000개 (70.0%)
Valid: 20,000개 (20.0%)
Test:  10,000개 (10.0%)

타겟 변수 분포:
Train - 0: 91.50%, 1: 8.50%
Valid - 0: 91.50%, 1: 8.50%
Test  - 0: 91.50%, 1: 8.50%

학습용 데이터 크기: (70000, 8), 정답: (70000,)
검증용 데이터 크기: (20000, 8), 정답: (20000,)
테스트용 데이터 크기: (10000, 8), 정답: (10000,)

학습 데이터 타깃 비율:
당뇨병 여부
0    0.915
1    0.085
Name: proportion, dtype: float64


# 데이터 전처리 

## 1. 결측치 처리

결측치가 없어서 안해도 됨

In [4]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)
print(df.isnull().sum())


결측치 확인
성별          0
나이          0
고혈압 여부      0
심장질환 여부     0
흡연 경험       0
BMI 지수      0
당화혈색소 수치    0
혈당 수치       0
당뇨병 여부      0
dtype: int64


## 2. 이상치 처리

- [나이], [흡연 경험] -> 범주형 변수(추후 다룸)
- [고혈압 여부], [심장질환 여부] -> 이진값이므로 제외 
- [나이] -> 최솟값 0.08, 최댓값 80으로 정상 범위이므로 제외 
- [당화혈색소 수치], [혈당 수치] -> "이상치 내부 당뇨 환자 분포"가 100% 이므로 이상치가 아닌 유의미한 데이터. 

-> 따라서 [BMI 지수]에 대해서만 이상치 처리 진행. 

In [5]:
df.describe()

,나이,고혈압 여부,심장질환 여부,BMI 지수,당화혈색소 수치,혈당 수치,당뇨병 여부
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


In [6]:
# 분석할 컬럼 목록
target_cols = ['BMI 지수', '당화혈색소 수치', '혈당 수치']

def analyze_outliers_detailed(df, col):
    """
    특정 컬럼에 대해 IQR 기반 이상치를 분석하고 리포트를 출력하는 함수
    """
    # 1) IQR 및 정상 범위 계산
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"■ [{col}]")
    print(f"1) 정상 범위: {lower_bound:.2f} ~ {upper_bound:.2f}")
    
    # 2) 이상치 데이터 발색 (필터링)
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    total_count = len(df)
    outlier_count = len(outliers)
    
    print(f"2) 이상치 개수: {outlier_count}개")
    
    # 3) 이상치 중 당뇨(Positive) 비율 분석
    if outlier_count > 0:
        diabetes_in_outliers = outliers[outliers['당뇨병 여부'] == 1]
        diabetes_count = len(diabetes_in_outliers)
        diabetes_ratio = (diabetes_count / outlier_count) * 100
        
        print(f"3) 이상치 내부 당뇨 환자 분포:")
        print(f"   - 당뇨 환자 수: {diabetes_count}명")
        print(f"   - 당뇨 비율: {diabetes_ratio:.2f}%")
        
        # 비교를 위해 전체 데이터의 당뇨 비율도 함께 출력하면 좋습니다
        total_diabetes_ratio = (df['당뇨병 여부'].sum() / total_count) * 100
    else:
        print("3) 발견된 이상치가 없습니다.")
        
    print("=" * 50)  # 구분선

# 실행
for col in target_cols:
    analyze_outliers_detailed(df, col)

■ [BMI 지수]
1) 정상 범위: 14.71 ~ 38.50
2) 이상치 개수: 7086개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 1478명
   - 당뇨 비율: 20.86%
■ [당화혈색소 수치]
1) 정상 범위: 2.70 ~ 8.30
2) 이상치 개수: 1315개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 1315명
   - 당뇨 비율: 100.00%
■ [혈당 수치]
1) 정상 범위: 11.50 ~ 247.50
2) 이상치 개수: 2038개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 2038명
   - 당뇨 비율: 100.00%


- [BMI 지수]가 높을수록 당뇨 여부에 영향이 있을 것으로 판단
- 따라서 정상 범위 최댓값보다 큰 이상치들을 상한값으로 대체
- 정상 범위 최솟값보다 작은 이상치들은 당뇨 여부에 큰 영향이 없을 것으로 판단하고 별도의 처리 X

In [7]:
bmi_col = "BMI 지수"

Q1 = X_train_raw[bmi_col].quantile(0.25)
Q3 = X_train_raw[bmi_col].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

print(f"✅ BMI 상한값(Upper Limit): {upper_limit:.2f}")

# 2. 학습 데이터(X_train_raw)에 적용: 상한값보다 큰 값은 상한값으로 대체 (Clipping)
X_train_raw[bmi_col] = X_train_raw[bmi_col].clip(upper=upper_limit)

# 3. 검증/테스트 데이터에도 '학습 데이터에서 구한 기준' 그대로 적용
X_valid_raw[bmi_col] = X_valid_raw[bmi_col].clip(upper=upper_limit)
X_test_raw[bmi_col] = X_test_raw[bmi_col].clip(upper=upper_limit)

print(">> 이상치 처리 완료: 상한값 초과 데이터를 상한값으로 대체했습니다.")

# (선택) 처리 후 최댓값이 상한값과 같아졌는지 확인
print(f"처리 후 Train BMI 최대값: {X_train_raw[bmi_col].max()}")
print(f"처리 후 Valid BMI 최대값: {X_valid_raw[bmi_col].max()}")
print(f"처리 후 Test BMI 최대값: {X_test_raw[bmi_col].max()}")

✅ BMI 상한값(Upper Limit): 38.43
>> 이상치 처리 완료: 상한값 초과 데이터를 상한값으로 대체했습니다.
처리 후 Train BMI 최대값: 38.43000000000001
처리 후 Valid BMI 최대값: 38.43000000000001
처리 후 Test BMI 최대값: 38.43000000000001


## [성별] 피처의 'Others' 값만 제거

In [8]:
print(f"삭제 전 크기: {X_train_raw.shape}")

# 성별이 'Other'가 아닌 것만 남기기
X_train_raw = X_train_raw[X_train_raw['성별'] != 'Other'].copy()
X_valid_raw = X_valid_raw[X_valid_raw['성별'] != 'Other'].copy()
X_test_raw = X_test_raw[X_test_raw['성별'] != 'Other'].copy()

# 삭제 후 y(정답지) 데이터도 인덱스 맞춰서 줄여줘야 함 (매우 중요!)
y_train = y_train[X_train_raw.index]
y_valid = y_valid[X_valid_raw.index]
y_test = y_test[X_test_raw.index]

print(f"삭제 후 크기: {X_train_raw.shape}")

# 2. 성별을 0과 1로 매핑 (이제 카테고리가 2개뿐이므로 순서 문제 없음)
# gender_map = {'Female': 0, 'Male': 1}

# X_train_raw['성별'] = X_train_raw['성별'].map(gender_map)
# X_valid_raw['성별'] = X_valid_raw['성별'].map(gender_map)
# X_test_raw['성별'] = X_test_raw['성별'].map(gender_map)

# 3. 데이터 타입 확인 (정수형으로 잘 바뀌었는지)
print("\n[변환 후 성별 분포]")
print(X_train_raw['성별'].value_counts())

삭제 전 크기: (70000, 8)
삭제 후 크기: (69985, 8)

[변환 후 성별 분포]
성별
Female    40947
Male      29038
Name: count, dtype: int64


# 모델 학습 및 예측

In [9]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support
import optuna


# =============================
# 0) 설정: "너무 오래 안 걸리게"
# =============================
RANDOM_STATE = 42
N_SPLITS = 5

# HPT 단계에서만 dev를 일부 샘플링(층화 유지)해서 속도 확보
# (너무 느리면 20000~40000 추천, 충분히 빠르면 None으로 풀기)
HPT_SAMPLE_SIZE = 30000

# Optuna 예산(시간/횟수)
N_TRIALS = 20
TIMEOUT_SEC = 60 * 10  # 10분 제한

EARLY_STOPPING_ROUNDS = 50
THREAD_COUNT = -1  # CPU 코어 최대 활용


# =============================
# 1) 데이터 준비(dev = train+valid)
# =============================
X_dev = pd.concat([X_train_raw, X_valid_raw], axis=0).reset_index(drop=True)
y_dev = pd.concat([y_train, y_valid], axis=0).reset_index(drop=True)

X_test = X_test_raw.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# 범주형: 인코딩 없이 원본 그대로 CatBoost에 맡김
cat_cols = X_dev.select_dtypes(include=["object", "category"]).columns.tolist()
cat_idx = [X_dev.columns.get_loc(c) for c in cat_cols]

# CatBoost에 안전하게 넣기 위해 문자열로 통일(새 범주도 문자열로 들어오게)
for c in cat_cols:
    X_dev[c] = X_dev[c].astype(str)
    X_test[c] = X_test[c].astype(str)

print("\n[Label distribution]")
print(f"train pos rate: {float(np.mean(y_train)):.4f} (count={int(y_train.sum())}/{len(y_train)})")
print(f"valid pos rate: {float(np.mean(y_valid)):.4f} (count={int(y_valid.sum())}/{len(y_valid)})")
print(f"test  pos rate: {float(np.mean(y_test)):.4f} (count={int(y_test.sum())}/{len(y_test)})")


# =============================
# 2) 유틸: PR-curve 기반 best threshold (그리드 스캔 제거)
# =============================
def prf_pos(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    return float(p), float(r), float(f1)

def best_threshold_from_pr_curve(y_true, proba_pos):
    # precision, recall: len = n_thresholds + 1
    # thresholds: len = n_thresholds, thresholds[i]는 precision[i+1], recall[i+1]에 대응
    precision, recall, thresholds = precision_recall_curve(y_true, proba_pos, pos_label=1)

    if len(thresholds) == 0:
        return 0.5, 0.0

    p = precision[1:]
    r = recall[1:]
    f1 = (2 * p * r) / (p + r + 1e-12)

    best_i = int(np.nanargmax(f1))
    return float(thresholds[best_i]), float(f1[best_i])

def print_prf(name, y_true, proba_pos, thr):
    y_pred = (proba_pos >= thr).astype(int)
    p, r, f1 = prf_pos(y_true, y_pred)

    print(f"\n[{name}] pos=1 threshold={thr:.4f}")
    print(f"precision: {p:.4f}")
    print(f"recall:    {r:.4f}")
    print(f"f1-score:  {f1:.4f}")

    true_pos_rate = float(np.mean(y_true))
    pred_pos_rate = float(np.mean(y_pred))
    print(f"true pos rate: {true_pos_rate:.4f} (count={int(np.sum(y_true))}/{len(y_true)})")
    print(f"pred pos rate: {pred_pos_rate:.4f} (count={int(np.sum(y_pred))}/{len(y_pred)})")

    q = np.quantile(proba_pos, [0.01, 0.5, 0.99])
    print(f"proba quantiles (1%,50%,99%): {q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}")


# =============================
# 3) HPT용 subsample(dev)
# =============================
def stratified_subsample(X, y, n, seed=42):
    if n is None or n >= len(X):
        return X, y

    rng = np.random.RandomState(seed)
    idx0 = np.where(y.values == 0)[0]
    idx1 = np.where(y.values == 1)[0]

    n1 = int(round(n * (len(idx1) / len(y))))
    n1 = min(n1, len(idx1))
    n0 = n - n1
    n0 = min(n0, len(idx0))

    pick0 = rng.choice(idx0, size=n0, replace=False)
    pick1 = rng.choice(idx1, size=n1, replace=False)
    pick = np.concatenate([pick0, pick1])
    rng.shuffle(pick)

    return X.iloc[pick].reset_index(drop=True), y.iloc[pick].reset_index(drop=True)

X_hpt, y_hpt = stratified_subsample(X_dev, y_dev, HPT_SAMPLE_SIZE, seed=RANDOM_STATE)
cat_idx_hpt = [X_hpt.columns.get_loc(c) for c in cat_cols]


# =============================
# 4) Optuna search space (파라미터 4개만)
#    - iterations: "최대 트리 수", 기본 1000 [page:2]
#    - learning_rate: gradient step 축소, 작을수록 더 많은 iterations 필요(시간 증가) [page:1]
#    - depth: 문서에서 보통 4~10이 최적, 6~10 권장 [page:1]
#    - l2_leaf_reg: L2 계수, 양수 허용, 기본 3.0 [page:2]
# =============================
def suggest_params(trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",

        # 4개만 튜닝
        "iterations": trial.suggest_int("iterations", 400, 1600),  # 기본 1000 포함 [page:2]
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),               # 문서 권장 구간 [page:1]
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),  # 기본 3.0 포함 [page:2]

        # 불균형: 자동 class weight (요구사항)
        "auto_class_weights": "Balanced",  # class_weights/scale_pos_weight와 같이 쓰면 안 됨 [page:2]

        # CPU/재현성/속도
        "task_type": "CPU",
        "random_seed": RANDOM_STATE,       # random_seed(random_state) [page:2]
        "thread_count": THREAD_COUNT,
        "allow_writing_files": False,
        "verbose": False,
    }
    return params


# =============================
# 5) Objective: 5-fold CV에서 "fold 내 best-threshold F1" 평균 최대화
#    + pruner 적용(느린 trial 빨리 중단)
# =============================
def objective(trial):
    params = suggest_params(trial)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    fold_f1s = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_hpt, y_hpt), start=1):
        X_tr, y_tr = X_hpt.iloc[tr_idx], y_hpt.iloc[tr_idx]
        X_va, y_va = X_hpt.iloc[va_idx], y_hpt.iloc[va_idx]

        tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx_hpt)
        va_pool = Pool(X_va, y_va, cat_features=cat_idx_hpt)

        model = CatBoostClassifier(**params)
        model.fit(
            tr_pool,
            eval_set=va_pool,
            use_best_model=True,                 # eval_metric 기준 best iteration만 남김 [page:1]
            early_stopping_rounds=EARLY_STOPPING_ROUNDS
        )

        proba_va = model.predict_proba(va_pool)[:, 1]
        thr, best_f1 = best_threshold_from_pr_curve(y_va.values, proba_va)
        fold_f1s.append(best_f1)

        trial.report(float(np.mean(fold_f1s)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_f1s))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

print("\n[Optuna best]")
print("CV mean best-F1:", study.best_value)
print("best params:", study.best_params)


# =============================
# 6) best params로 dev 전체에서 OOF 확률 -> global best threshold 결정
# =============================
best_params = suggest_params(optuna.trial.FixedTrial(study.best_params))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(X_dev), dtype=float)

for tr_idx, va_idx in skf.split(X_dev, y_dev):
    X_tr, y_tr = X_dev.iloc[tr_idx], y_dev.iloc[tr_idx]
    X_va, y_va = X_dev.iloc[va_idx], y_dev.iloc[va_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    va_pool = Pool(X_va, y_va, cat_features=cat_idx)

    model = CatBoostClassifier(**best_params)
    model.fit(
        tr_pool,
        eval_set=va_pool,
        use_best_model=True,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )

    oof_proba[va_idx] = model.predict_proba(va_pool)[:, 1]

best_thr, best_oof_f1 = best_threshold_from_pr_curve(y_dev.values, oof_proba)
print(f"\n[DEV-OOF] best threshold={best_thr:.4f}, best f1={best_oof_f1:.4f}")
print_prf("DEV (OOF)", y_dev.values, oof_proba, best_thr)


# =============================
# 7) dev 전체로 최종 학습 -> test 1회 평가
# =============================
dev_pool = Pool(X_dev, y_dev, cat_features=cat_idx)
test_pool = Pool(X_test, y_test, cat_features=cat_idx)

final_model = CatBoostClassifier(**best_params)
final_model.fit(dev_pool, use_best_model=False)

test_proba = final_model.predict_proba(test_pool)[:, 1]
print_prf("TEST", y_test.values, test_proba, best_thr)


c:\Users\ghkda\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-20 16:16:58,043] A new study created in memory with name: no-name-f3064632-383c-4d36-8cd8-12dd42b04124



[Label distribution]
train pos rate: 0.0850 (count=5950/69985)
valid pos rate: 0.0850 (count=1700/19997)
test  pos rate: 0.0850 (count=850/10000)


Best trial: 0. Best value: 0.807213:   5%|▌         | 1/20 [00:31<10:02, 31.71s/it, 31.71/600 seconds]

[I 2026-02-20 16:17:29,754] Trial 0 finished with value: 0.8072134787949716 and parameters: {'iterations': 849, 'learning_rate': 0.17254716573280354, 'depth': 9, 'l2_leaf_reg': 7.6611007077713635}. Best is trial 0 with value: 0.8072134787949716.


Best trial: 1. Best value: 0.812206:  10%|█         | 2/20 [02:09<21:07, 70.40s/it, 129.20/600 seconds]

...

[I 2026-02-20 16:26:24,837] Trial 5 finished with value: 0.8125586009226815 and parameters: {'iterations': 1134, 'learning_rate': 0.01518747922672247, 'depth': 6, 'l2_leaf_reg': 3.4766491505926194}. Best is trial 5 with value: 0.8125586009226815.


Best trial: 5. Best value: 0.812559:  35%|███▌      | 7/20 [09:37<16:17, 75.17s/it, 577.14/600 seconds] 

[I 2026-02-20 16:26:35,190] Trial 6 pruned. 


Best trial: 5. Best value: 0.812559:  40%|████      | 8/20 [10:40<16:01, 80.11s/it, 640.86/600 seconds]


[I 2026-02-20 16:27:38,896] Trial 7 pruned. 

[Optuna best]
CV mean best-F1: 0.8125586009226815
best params: {'iterations': 1134, 'learning_rate': 0.01518747922672247, 'depth': 6, 'l2_leaf_reg': 3.4766491505926194}

[DEV-OOF] best threshold=0.8607, best f1=0.8070

[DEV (OOF)] pos=1 threshold=0.8607
precision: 0.9410
recall:    0.7063
f1-score:  0.8069
true pos rate: 0.0850 (count=7650/89982)
pred pos rate: 0.0638 (count=5742/89982)
proba quantiles (1%,50%,99%): 0.0000, 0.0033, 0.9999

[TEST] pos=1 threshold=0.8607
precision: 0.9392
recall:    0.7082
f1-score:  0.8075
true pos rate: 0.0850 (count=850/10000)
pred pos rate: 0.0641 (count=641/10000)
proba quantiles (1%,50%,99%): 0.0000, 0.0024, 1.0000
